# 85 - Resultado da revisão manual dos clusters

**Pergunta:** quantos clusters de cada configuração da Tabela 4 do TCC II foram considerados coerentes na revisão manual cega? Como isso se relaciona com a silhueta e a cobertura?

**Origem dos dados:**

- `reports/tcc2_avaliacao/revisao_manual.xlsx`, aba `Avaliacao`: avaliações preenchidas (coerente, parcialmente coerente, incoerente). A coluna `avaliador` indica quem avaliou: `autor` ou `claude` (avaliação assistida por IA, revisável pelo autor). Linhas sem avaliador são tratadas como `autor`, porque foram preenchidas por ele depois da etapa assistida.
- `reports/tcc2_avaliacao/revisao_chave.csv`: correspondência entre código, experimento, cluster, estrato de tamanho e pureza NCM, gerada pelo notebook 84.
- `reports/tcc2_selecionados/tcc2_experimentos_selecionados.xlsx`, aba `Resumo`: silhueta (Tabela 4) e cobertura, gerada pelo notebook 83.
- `reports/tcc2_avaliacao/metricas_ncm.xlsx`, aba `Clusters`: tamanho de todos os clusters de cada experimento, usado para refazer os tercis e ponderar a estimativa por descrições.

**Amostra:** 7 clusters por tercil de tamanho em cada experimento (todos os 11 no MPNet + DBSCAN). Como os tercis têm o mesmo peso na amostra, os percentuais de clusters descrevem a amostra estratificada. Além deles, o notebook estima a parcela de **descrições** que está em clusters coerentes: em cada tercil, a taxa de coerência da amostra é ponderada pelo tamanho dos clusters sorteados e aplicada ao total de descrições do tercil. Multiplicada pela cobertura, essa estimativa indica quantas das 20.000 descrições ficaram em clusters coerentes.

**Saída:** `reports/tcc2_avaliacao/resultado_revisao.xlsx`, regravado a cada execução. Este notebook apenas lê a planilha de revisão, sem alterá-la.

In [1]:
from pathlib import Path
import logging

import pandas as pd
from IPython.display import display
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook85")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## Configurações

In [2]:
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não encontrei a raiz do repositório com data/processed/.")

REVIEW_PATH = "reports/tcc2_avaliacao/revisao_manual.xlsx"
KEY_PATH = "reports/tcc2_avaliacao/revisao_chave.csv"
SUMMARY_PATH = "reports/tcc2_selecionados/tcc2_experimentos_selecionados.xlsx"
NCM_CLUSTERS_PATH = "reports/tcc2_avaliacao/metricas_ncm.xlsx"
EXPECTED_ROWS = 20_000
OUTPUT_PATH = "reports/tcc2_avaliacao/resultado_revisao.xlsx"

REVIEW_OPTIONS = ["coerente", "parcialmente coerente", "incoerente"]
DEFAULT_REVIEWER = "autor"
STRATA = ["pequeno", "medio", "grande"]

# Clusters que concentram uma parcela grande das descrições agrupadas e ficaram fora da
# amostra distorcem a estimativa por descrições. Eles devem ser classificados aqui, com motivo.
DOMINANT_SHARE = 0.20
DOMINANT_CLUSTERS = {
    ("01 TFIDF KMeans", 0): (
        "incoerente",
        "13.885 descrições (69,4%), incluindo todas as 10.796 sem atributos TF-IDF (vetor nulo); "
        "não forma um tipo de produto por construção.",
    ),
}

## Funções

In [3]:
def count_reviews(frame, by):
    """Conta avaliações por grupo e acrescenta totais e percentuais."""
    counts = (
        frame.groupby(by, observed=True)["avaliacao"].value_counts().unstack(fill_value=0)
        .reindex(columns=REVIEW_OPTIONS, fill_value=0)
    )
    counts["avaliados"] = counts[REVIEW_OPTIONS].sum(axis=1)
    counts["coerente_pct"] = (counts["coerente"] / counts["avaliados"] * 100).round(1)
    counts["coerente_ou_parcial_pct"] = (
        (counts["coerente"] + counts["parcialmente coerente"]) / counts["avaliados"] * 100
    ).round(1)
    return counts.reset_index()


def estimate_coherent_descriptions(sampled, all_clusters):
    """Estima, por experimento, a fração das descrições agrupadas que está em clusters coerentes.

    Refaz os tercis de tamanho do notebook 84 sobre todos os clusters, calcula em cada
    tercil a coerência da amostra ponderada pelo tamanho e pondera os tercis pelo total
    de descrições que contêm. Clusters dominantes fora da amostra entram com a
    classificação de DOMINANT_CLUSTERS, e não pela taxa do tercil.
    """
    rows = []
    for experiment, group in all_clusters.groupby("experimento"):
        sizes = group.set_index("cluster")["tamanho"].sort_index()
        strata = pd.qcut(sizes.rank(method="first"), 3, labels=STRATA)
        sample = sampled.loc[sampled["experimento"].eq(experiment)].copy()
        dominant = sizes[sizes.ge(DOMINANT_SHARE * sizes.sum()) & ~sizes.index.isin(sample["cluster"])]
        unclassified = [cluster for cluster in dominant.index if (experiment, cluster) not in DOMINANT_CLUSTERS]
        if unclassified:
            raise ValueError(f"{experiment}: classifique os clusters dominantes fora da amostra {unclassified}")
        dominant_labels = pd.Series({c: DOMINANT_CLUSTERS[(experiment, c)][0] for c in dominant.index}, dtype=object)
        dominant_strict = dominant[dominant_labels.eq("coerente")].sum()
        dominant_tolerant = dominant[dominant_labels.ne("incoerente")].sum()
        stratum_totals = sizes.drop(dominant.index).groupby(strata.drop(dominant.index), observed=True).sum()
        sample["peso_coerente"] = sample["tamanho"] * sample["avaliacao"].eq("coerente")
        sample["peso_tolerante"] = sample["tamanho"] * sample["avaliacao"].ne("incoerente")
        rates = sample.groupby("estrato", observed=True)[["peso_coerente", "peso_tolerante", "tamanho"]].sum()
        strict = ((rates["peso_coerente"] / rates["tamanho"] * stratum_totals).sum() + dominant_strict) / sizes.sum()
        tolerant = ((rates["peso_tolerante"] / rates["tamanho"] * stratum_totals).sum() + dominant_tolerant) / sizes.sum()
        rows.append({
            "experimento": experiment,
            "agrupadas_em_coerentes_pct": round(strict * 100, 1),
            "agrupadas_em_coerentes_ou_parciais_pct": round(tolerant * 100, 1),
            "agrupadas": int(sizes.sum()),
            "clusters_dominantes_fora_amostra": ", ".join(
                f"{c} ({int(dominant[c])}, {dominant_labels[c]})" for c in dominant.index
            ),
        })
    estimate = pd.DataFrame(rows)
    estimate["amostra_em_coerentes_pct"] = (
        estimate["agrupadas_em_coerentes_pct"] * estimate["agrupadas"] / EXPECTED_ROWS
    ).round(1)
    return estimate.drop(columns="agrupadas")


def write_sheet(writer, frame, sheet_name):
    """Grava uma aba como Tabela do Excel com cabeçalho fixo."""
    frame.to_excel(writer, sheet_name=sheet_name, index=False)
    worksheet = writer.sheets[sheet_name]
    last_cell = f"{get_column_letter(frame.shape[1])}{len(frame) + 1}"
    table = Table(displayName=f"t_{sheet_name.lower()}", ref=f"A1:{last_cell}")
    table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
    worksheet.add_table(table)
    worksheet.freeze_panes = "A2"
    for index, column in enumerate(frame.columns, start=1):
        width = max(len(str(column)), frame[column].astype(str).str.len().quantile(0.95))
        worksheet.column_dimensions[get_column_letter(index)].width = min(max(width + 2, 10), 70)

## Execução

### 1. Carregar avaliações, chave e resumo

In [4]:
reviews = pd.read_excel(PROJECT_ROOT / REVIEW_PATH, sheet_name="Avaliacao")
keys = pd.read_csv(PROJECT_ROOT / KEY_PATH)
summary = pd.read_excel(PROJECT_ROOT / SUMMARY_PATH, sheet_name="Resumo")
all_clusters = pd.read_excel(PROJECT_ROOT / NCM_CLUSTERS_PATH, sheet_name="Clusters")

missing = reviews.loc[reviews["avaliacao"].isna(), "codigo"].tolist()
invalid = sorted(set(reviews["avaliacao"].dropna()) - set(REVIEW_OPTIONS))
if missing or invalid:
    raise ValueError(f"Avaliações vazias: {missing}; valores inválidos: {invalid}")
if set(reviews["codigo"]) != set(keys["codigo"]):
    raise ValueError("Os códigos da revisão e da chave não coincidem.")

reviews["avaliador"] = reviews["avaliador"].fillna(DEFAULT_REVIEWER)
clusters = keys.merge(reviews[["codigo", "avaliacao", "observacao", "avaliador"]], on="codigo")
clusters["estrato"] = pd.Categorical(clusters["estrato"], categories=STRATA, ordered=True)
logger.info("%d clusters avaliados em %d experimentos", len(clusters), clusters["experimento"].nunique())
display(clusters["avaliador"].value_counts())

2026-10-26 12:01:38,254 INFO 179 clusters avaliados em 9 experimentos


avaliador
claude    150
autor      29
Name: count, dtype: int64

### 2. Resultado por experimento

Tabela principal: silhueta e cobertura da Tabela 4 ao lado das contagens da revisão manual e da estimativa ponderada por descrições.

- `coerente_pct`: parcela dos clusters sorteados avaliados como coerentes.
- `agrupadas_em_coerentes_pct`: estimativa da parcela das descrições agrupadas que está em clusters coerentes.
- `amostra_em_coerentes_pct`: a estimativa anterior multiplicada pela cobertura, ou seja, a parcela das 20.000 descrições que ficou em clusters coerentes.
- `clusters_dominantes_fora_amostra`: clusters com pelo menos 20% das descrições agrupadas que não foram sorteados e entraram na estimativa pela classificação de `DOMINANT_CLUSTERS`. No TF-IDF + K-Means, o cluster 0 reúne todas as descrições sem atributos TF-IDF e é tratado como incoerente.

In [5]:
by_experiment = count_reviews(clusters, "experimento").merge(
    summary[["aba", "representacao", "algoritmo", "destaque", "clusters", "silhueta_tabela4", "cobertura_pct"]]
    .rename(columns={"aba": "experimento", "clusters": "clusters_total", "silhueta_tabela4": "silhueta"}),
    on="experimento",
).merge(estimate_coherent_descriptions(clusters, all_clusters), on="experimento")
by_experiment = by_experiment[[
    "experimento", "representacao", "algoritmo", "destaque", "clusters_total", "silhueta", "cobertura_pct",
    "avaliados", *REVIEW_OPTIONS, "coerente_pct", "coerente_ou_parcial_pct",
    "agrupadas_em_coerentes_pct", "agrupadas_em_coerentes_ou_parciais_pct", "amostra_em_coerentes_pct",
    "clusters_dominantes_fora_amostra",
]]
display(by_experiment)

,experimento,representacao,algoritmo,destaque,clusters_total,silhueta,cobertura_pct,avaliados,coerente,parcialmente coerente,incoerente,coerente_pct,coerente_ou_parcial_pct,agrupadas_em_coerentes_pct,agrupadas_em_coerentes_ou_parciais_pct,amostra_em_coerentes_pct,clusters_dominantes_fora_amostra
0,01 TFIDF KMeans,TF-IDF,K-Means,1º do ranking,200,0.2099,100.000,21,17,2,2,81.0,90.5,24.4,28.2,24.4,"0 (13885, incoerente)"
1,02 MPNet KMeans,MPNet,K-Means,1º do ranking,200,0.1165,100.000,21,6,6,9,28.6,57.1,19.5,41.5,19.5,
2,03 TFIDF DBSCAN,TF-IDF,DBSCAN,1º do ranking,734,0.9996,36.085,21,21,0,0,100.0,100.0,100.0,100.0,36.1,
3,04 MPNet DBSCAN,MPNet,DBSCAN,1º do ranking,11,0.9753,0.665,11,11,0,0,100.0,100.0,100.0,100.0,0.7,
4,05 TFIDF HDBSCAN 1o,TF-IDF,HDBSCAN,1º do ranking,736,0.5189,50.915,21,20,0,1,95.2,95.2,95.9,95.9,48.8,
5,06 TFIDF HDBSCAN A,TF-IDF,HDBSCAN,Candidata A,101,0.7127,20.005,21,17,4,0,81.0,100.0,87.2,100.0,17.4,
6,07 TFIDF HDBSCAN B,TF-IDF,HDBSCAN,Candidata B,381,0.5656,39.735,21,19,2,0,90.5,100.0,91.6,100.0,36.4,
7,08 MPNet HDBSCAN,MPNet,HDBSCAN,1º do ranking,123,0.4710,16.905,21,18,1,2,85.7,90.5,90.1,91.2,15.2,
8,09 Hibrido HDBSCAN,TF-IDF + MPNet (alpha=0.2),HDBSCAN,1º do ranking,982,0.3487,48.355,21,18,3,0,85.7,100.0,88.3,100.0,42.7,


### 3. Detalhamentos

- Por estrato de tamanho: mostra se os clusters grandes são mais misturados que os pequenos.
- Por avaliador: mostra quanto da avaliação foi feita pelo autor e quanto foi assistida.
- Pureza NCM média por avaliação: se clusters avaliados como coerentes também tiverem pureza NCM maior, as duas evidências se reforçam.

In [6]:
by_stratum = count_reviews(clusters, ["experimento", "estrato"])
by_reviewer = count_reviews(clusters, "avaliador")
purity_by_review = (
    clusters.groupby("avaliacao")["pureza"].agg(clusters="size", pureza_ncm_media="mean", pureza_ncm_mediana="median")
    .reindex(REVIEW_OPTIONS).round(3).reset_index()
)
display(by_stratum.pivot(index="experimento", columns="estrato", values="coerente_pct"))
display(by_reviewer)
display(purity_by_review)

estrato,pequeno,medio,grande
experimento,,,
01 TFIDF KMeans,71.4,85.7,85.7
02 MPNet KMeans,42.9,42.9,0.0
03 TFIDF DBSCAN,100.0,100.0,100.0
04 MPNet DBSCAN,100.0,100.0,100.0
05 TFIDF HDBSCAN 1o,100.0,85.7,100.0
06 TFIDF HDBSCAN A,71.4,71.4,100.0
07 TFIDF HDBSCAN B,100.0,85.7,85.7
08 MPNet HDBSCAN,71.4,100.0,85.7
09 Hibrido HDBSCAN,100.0,71.4,85.7


avaliacao,avaliador,coerente,parcialmente coerente,incoerente,avaliados,coerente_pct,coerente_ou_parcial_pct
0,autor,24,3,2,29,82.8,93.1
1,claude,123,15,12,150,82.0,92.0


,avaliacao,clusters,pureza_ncm_media,pureza_ncm_mediana
0,coerente,147,0.797,0.875
1,parcialmente coerente,18,0.461,0.402
2,incoerente,14,0.281,0.255


### 4. Gravar o resultado

**Grava** `reports/tcc2_avaliacao/resultado_revisao.xlsx`, com as abas `Experimentos`, `Estratos`, `Avaliadores`, `Pureza_NCM` e `Clusters` (detalhe de cada cluster avaliado, já identificado pelo experimento).

In [7]:
output_path = PROJECT_ROOT / OUTPUT_PATH
detail = clusters.sort_values(["experimento", "estrato", "cluster"])[[
    "experimento", "estrato", "cluster", "tamanho", "codigo", "avaliacao", "observacao",
    "avaliador", "ncm_majoritario", "pureza",
]]
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    write_sheet(writer, by_experiment, "Experimentos")
    write_sheet(writer, by_stratum, "Estratos")
    write_sheet(writer, by_reviewer, "Avaliadores")
    write_sheet(writer, purity_by_review, "Pureza_NCM")
    write_sheet(writer, detail.round({"pureza": 3}), "Clusters")
logger.info("Resultado gravado em %s", output_path)

2026-10-26 12:01:38,422 INFO Resultado gravado em C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\tcc2_avaliacao\resultado_revisao.xlsx


## Interpretação

- `amostra_em_coerentes_pct` é o número mais simples para o texto, porque junta coerência e cobertura: a parcela das 20.000 descrições que ficou em clusters coerentes. É uma estimativa a partir da amostra; com 7 clusters por tercil, um único cluster grande pode mudar bastante o valor.
- `coerente_pct` responde à pergunta por cluster: a parcela dos clusters sorteados que reúne um único tipo de produto. `coerente_ou_parcial_pct` é a versão tolerante, que inclui grupos amplos ou com poucas exceções.
- Leia sempre junto da cobertura: uma configuração com muitos clusters coerentes, mas pouca cobertura, organiza bem uma parcela pequena das descrições.
- Alguns clusters coerentes repetem o mesmo produto, variando apenas lote, nota fiscal ou quantidade (veja `observacao`). Eles são coerentes de forma trivial e podem ser mencionados à parte.
- Parte das avaliações foi assistida por IA (`avaliador = claude`) e revisada pelo autor. Isso deve ser declarado na metodologia.